# 06 - Analyse

Stage 8. Do the audits agree, model by model? Instance-level rank correlation of the normalized oracle gap, pass/fail disagreement, the patterns the hypotheses name, and the confirmatory mixed models -- all from Stage 7's calibration tables, so every number traces to a record.

**Attach** `forgetcheck-artifacts` and `forgetcheck-stage6`. **CPU only**: set Accelerator to None. A few minutes end to end.


In [ ]:
# --- clone the repo at a PINNED commit ------------------------------------------------------
# Clone rather than `pip install git+...`. A wheel would contain only src/forgetcheck/, but the
# CLI also needs configs/ (the metric registry, seed streams, audit protocols) and
# data/memorization/ (the RUM scores -- 400 KB, committed precisely so a fresh session does not
# have to re-download 2 GB from Google Drive).
#
# Pinning is what makes provenance work: every record this session writes carries this commit,
# so any result can be traced back to the exact code that produced it.
REPO   = "https://github.com/hyperreal2005/Minor-Project.git"
COMMIT = "main"          # <-- pin to a sha for real runs, e.g. "a1b2c3d"

import os
from pathlib import Path

os.chdir("/kaggle/working")
if not Path("Minor-Project").exists():
    !git clone --quiet $REPO
%cd /kaggle/working/Minor-Project
# Check out the FETCHED ref, not the local branch. In a session that already has the clone,
# `git checkout main` stays on the old local main -- `fetch` updates only origin/main -- so a
# re-run after a push silently kept executing the previous code, and the only sign was the
# "pinned at" line below. `origin/<branch>` follows the push; a sha falls through to the second
# form. Compare the printed sha with the commit you pushed.
!git fetch --quiet --all
!git checkout --quiet --detach origin/$COMMIT 2>/dev/null || git checkout --quiet --detach $COMMIT
!git log -1 --format="pinned at %h  %s"
!pip install -q -e .


In [ ]:
import importlib
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/Minor-Project")

# Make the package importable *in this kernel*.
#
# `pip install -e .` writes a .pth file into site-packages, and .pth files are only processed at
# interpreter startup. The kernel was already running when the previous cell installed, so
# sys.path never picked it up and `import forgetcheck` fails with ModuleNotFoundError. The
# `!forgetcheck` CLI calls below are unaffected -- each spawns a fresh Python that does read the
# .pth -- which makes this failure look stranger than it is.
#
# Adding src/ directly is deterministic and avoids making anyone restart the kernel.
SRC = str(REPO_DIR / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()

# `!` cells run in a subshell, which inherits this. Belt and braces: the console script should
# already be on PATH after the editable install, but if it is not, PYTHONPATH keeps
# `python -m forgetcheck.cli` working as a fallback.
import os
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")

import forgetcheck
print("forgetcheck imported from:", Path(forgetcheck.__file__).parent)

import shutil

# CIFAR-10 downloads at 100-130 kB/s on Kaggle -- 20 to 30 minutes, repeated on every session and
# every account. Attaching it as a Dataset (Add Input -> Datasets) skips that entirely:
# torchvision checks the md5s of the extracted folder and only downloads if it is missing or
# corrupt. 00_verify_setup.ipynb has a cell that creates the dataset once.
#
# Kaggle's mount layout varies with how a dataset was uploaded -- it may sit at
# /kaggle/input/<slug>/, or nested as /kaggle/input/datasets/<user>/<slug>/, or one level deeper
# again if the dataset was created from a notebook's output directory. So SEARCH for the folder
# rather than assume a path (`**/` matches at any depth, including directly under /kaggle/input),
# and then verify the copy actually landed. An earlier version of this cell
# used `cp ... 2>/dev/null || true` followed by an unconditional success message: a failed copy
# reported success and CIFAR silently re-downloaded anyway, costing ~28 minutes while the output
# claimed otherwise. Never report an outcome that was not checked.
def _restore(name, dest):
    "Find directory `name` anywhere under /kaggle/input and copy it to `dest`."
    dest = Path(dest)
    if dest.is_dir():
        print(f"{name}: already present")
        return True
    found = sorted(Path("/kaggle/input").glob(f"**/{name}"))
    if not found:
        return False
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(found[0], dest, dirs_exist_ok=True)
    print(f"{name}: copied from {found[0]}")
    return True

def _restore_all(name, dest):
    # Merge EVERY directory called `name` under /kaggle/input into `dest`, by symlink.
    #
    # `_restore` above takes the first match, which is right for stages 3-5: each account
    # needs one previous dataset. Stage 6 is different -- an account needs its own Stage 5
    # shard, the Stage 3 oracles and originals, and the Stage 4 shadows, which live in three
    # datasets. Only a merge gives the audit runner all of them under one store root.
    #
    # Symlinks, not copies: /kaggle/input is a read-only mount on a different filesystem, so
    # hard links are impossible and copies would move ~8 GB into the 19.5 GB working quota
    # for nothing. torch.load and Path.is_file() follow symlinks transparently. First match
    # wins for a file present in several datasets (the same run computed once; the pilot's
    # duplicate scrub runs are the only such case), and the count is printed.
    #
    # (Comments, not a docstring: this function lives inside a triple-quoted notebook
    # template, and a nested triple quote ends the template early.)
    dest = Path(dest)
    found = sorted(Path("/kaggle/input").glob(f"**/{name}"))
    found = [f for f in found if f.is_dir()]
    if not found:
        return False
    linked = dup = empty = 0
    empty_example = None
    empty_parquet = []
    for src_root in found:
        n_here = 0
        for src in src_root.rglob("*"):
            if not src.is_file():
                continue
            # Count zero-byte files, but say only what is known. `.gitkeep` and friends are
            # empty by design; a `.parquet` of zero bytes is a lost record shard; a `.npz` of
            # zero bytes is a cache the runner recomputes. One empty oracle activation file
            # once failed all 30 targets of a condition, so it is worth seeing at restore time.
            if src.stat().st_size == 0 and not src.name.startswith("."):
                empty += 1
                empty_example = empty_example or src
                if src.suffix == ".parquet":
                    empty_parquet.append(src.name)
            target = dest / src.relative_to(src_root)
            if target.exists() or target.is_symlink():
                dup += 1
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            os.symlink(src, target)
            linked += 1
            n_here += 1
        # The mount layout under /kaggle/input is Kaggle's to decide and has changed before.
        # Print the real path rather than letting anyone assume one.
        print(f"{name}: {n_here:5d} files linked from {src_root}")
    print(f"{name}: linked {linked} files from {len(found)} dataset(s)"
          + (f", {dup} already present" if dup else ""))
    if empty:
        print(f"!! {name}: {empty} zero-byte file(s), e.g. {empty_example}")
        if empty_parquet:
            print(f"!! {len(empty_parquet)} of them are RECORD SHARDS and their rows are lost "
                  f"in this version -- attach an earlier version. e.g. {empty_parquet[0]}")
        else:
            print("!! None are record shards. Empty cache files are recomputed and overwritten "
                  "by the audit runner; nothing is lost.")
    return True

# Restore CIFAR-10 by locating its *contents*, not its folder name.
#
# Kaggle does not necessarily preserve the directory that was uploaded: the batches may end up
# inside `cifar-10-batches-py/`, or flattened straight to the dataset root. Searching for the
# folder name therefore reports "not found" while the data sits one level up in plain view --
# which is exactly what happened here, and cost a 25-minute re-download.
#
# So anchor on a file that must exist (`test_batch`) and take whatever directory contains it.
# That handles both layouts, and any future one.
def _restore_cifar(dest):
    dest = Path(dest)
    if dest.is_dir() and len(list(dest.glob("*_batch*"))) == 6:
        print("cifar-10: already present")
        return True
    hits = sorted(Path("/kaggle/input").glob("**/test_batch"))
    if not hits:
        return False
    src = hits[0].parent
    dest.mkdir(parents=True, exist_ok=True)
    for p in src.iterdir():
        if p.is_file():
            shutil.copy2(p, dest / p.name)
    print(f"cifar-10: copied from {src}")
    return True

CIFAR_DEST = REPO_DIR / "data" / "cifar-10-batches-py"
if not _restore_cifar(CIFAR_DEST):
    print("!! no CIFAR-10 batches found under /kaggle/input -- it will DOWNLOAD (~25 min)")
    print("!! attached:", [p.name for p in sorted(Path("/kaggle/input").glob("*"))] or "(none)")

# Verify rather than trust: torchvision needs 5 training batches plus test_batch.
if CIFAR_DEST.is_dir():
    n = len(list(CIFAR_DEST.glob("*_batch*")))
    print(f"   {n}/6 batch files{'' if n == 6 else '  <-- INCOMPLETE, will re-download'}")

# Previous artefacts, so runs another session already finished are skipped rather than repeated.
for _name in ("artifacts", "results"):
    if not _restore_all(_name, REPO_DIR / _name):
        print(f"{_name}: none attached - starting fresh")

import torch
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("!! running on CPU. Set Settings -> Accelerator -> GPU.")
    print("!! On CPU one 30-epoch training run takes ~4.8 hours instead of ~12 minutes.")

CLI = "forgetcheck" if shutil.which("forgetcheck") else f"{sys.executable} -m forgetcheck.cli"
print("cli:", CLI)

# Stage 6's first pass named relearning-anchor shards without the condition, and the five clean
# base models serve seven conditions each, so those files hold a collided, superseded subset.
# The per-condition shards (`--relearn-anchor-<cond>`) replaced them; the old ones are removed
# so they cannot duplicate rows in a snapshot. They are symlinks here; the source is untouched.
_stale = list((REPO_DIR / "results").rglob("*--relearn-anchor.parquet"))
for _f in _stale:
    _f.unlink()
if _stale:
    print(f"removed {len(_stale)} superseded first-pass anchor shard(s)")


## Step 1 - calibrate

Rebuilds Stage 7's tables from the records (about a minute). Its full report was read in Stage 7; only the lines Stage 8 needs are shown, and the whole report is kept in the download.

In [ ]:
!{CLI} --root . calibrate > calibrate_report.txt
!grep -E 'records:|recovered|available for|relearning-only' calibrate_report.txt


## Step 2 - analyse

In [ ]:
!{CLI} --root . analyse


## Step 3 - one file to download

In the right-hand panel, under **Output** (`/kaggle/working`), open the menu next to `forgetcheck_stage8_results.zip` and download it. It holds only result tables (a few MB).

In [ ]:
# One small file holding every Stage 7 and Stage 8 table: result tables only, no checkpoints.
import zipfile
from pathlib import Path

bundle = Path("/kaggle/working/forgetcheck_stage8_results.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for sub in ("calibration", "analysis"):
        for p in sorted((REPO_DIR / "results" / sub).glob("*.parquet")):
            z.write(p, f"results/{sub}/{p.name}")
    z.write("calibrate_report.txt", "results/calibrate_report.txt")
print(f"{bundle}  ({bundle.stat().st_size / 1e6:.1f} MB)")
